# Fuse Lidar + Camera: GraphDECO 3DGS Colab T4 Smoke Test

Use this notebook in Google Colab with a T4 GPU. It avoids uploading a `.py` file. You only upload the packaged dataset ZIP when prompted.

Success means the official GraphDECO trainer loads our COLMAP-style export and reaches iteration 300. This is **not** a visual-quality benchmark yet.

In [ ]:
# Cell 1 — GPU sanity check
# Runtime -> Change runtime type -> T4 GPU before running this.
!nvidia-smi

import torch
print('torch:', torch.__version__)
print('torch cuda:', torch.version.cuda)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('CUDA is not available. Change runtime type to T4 GPU and rerun.')

In [ ]:
# Cell 2 — Upload the packaged Fuse dataset ZIP
from google.colab import files
from pathlib import Path

print('Upload: 20260625T214456Z-steady-undistorted-graphdeco.zip')
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No dataset ZIP uploaded')
ZIP_PATH = Path('/content') / next(iter(uploaded.keys()))
print('Dataset ZIP:', ZIP_PATH)

In [ ]:
# Cell 3 — Clone GraphDECO and install CUDA extensions
# The important fix is --no-build-isolation. Without it, pip can hide Colab's installed torch.
%cd /content
!rm -rf gaussian-splatting
!git clone https://github.com/graphdeco-inria/gaussian-splatting --recursive
%cd /content/gaussian-splatting

!python -m pip install -q --upgrade pip setuptools wheel
!python -m pip install -q plyfile tqdm opencv-python joblib ninja

!python - <<'PY'
import torch
print('torch', torch.__version__)
print('torch cuda', torch.version.cuda)
print('cuda available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device', torch.cuda.get_device_name(0))
PY

!nvcc --version
!python -m pip install -v --no-build-isolation ./submodules/diff-gaussian-rasterization
!python -m pip install -v --no-build-isolation ./submodules/simple-knn
# Optional; training falls back if this fails.
!python -m pip install -v --no-build-isolation ./submodules/fused-ssim || true

In [ ]:
# Cell 4 — Unpack and validate the dataset shape
import json
import shutil
import zipfile
from pathlib import Path

dataset_root = Path('/content/fuse_graphdeco_dataset') / ZIP_PATH.stem
shutil.rmtree(dataset_root, ignore_errors=True)
dataset_root.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH) as archive:
    archive.extractall(dataset_root)

required = [
    dataset_root / 'images',
    dataset_root / 'sparse' / '0' / 'cameras.bin',
    dataset_root / 'sparse' / '0' / 'images.bin',
    dataset_root / 'sparse' / '0' / 'points3D.bin',
    dataset_root / 'sparse' / '0' / 'points3D.ply',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('\n'.join(missing))

images = sorted((dataset_root / 'images').glob('*'))
print('dataset_root:', dataset_root)
print('image count:', len(images))
print('sparse files:', sorted(path.name for path in (dataset_root / 'sparse' / '0').glob('*')))

report_path = dataset_root / 'graphdeco_input_check.json'
if report_path.exists():
    report = json.loads(report_path.read_text())
    print('ready:', report.get('ready_for_graphdeco_loader'))
    print('points:', report.get('sparse_points', {}).get('count'))
    print('track refs:', report.get('sparse_points', {}).get('track_reference_count'))

In [ ]:
# Cell 5 — Run the 300-iteration GraphDECO smoke test
import shlex
import subprocess
from pathlib import Path

out_dir = Path('/content/fuse_3dgs_output') / f'{dataset_root.name}-smoke'
out_dir.parent.mkdir(parents=True, exist_ok=True)

cmd = [
    'python', 'train.py',
    '-s', str(dataset_root),
    '-m', str(out_dir),
    '--images', 'images',
    '--resolution', '8',
    '--iterations', '300',
    '--test_iterations', '300',
    '--save_iterations', '300',
    '--data_device', 'cpu',
    '--disable_viewer',
]
print(' '.join(shlex.quote(part) for part in cmd))
subprocess.run(cmd, cwd='/content/gaussian-splatting', check=True)
print('PASS: training reached iteration 300')
print('output:', out_dir)

In [ ]:
# Cell 6 — Download output archive
from google.colab import files
import shutil

for path in sorted(out_dir.rglob('*'))[:100]:
    print(path.relative_to(out_dir))

archive_path = shutil.make_archive(str(out_dir), 'zip', out_dir)
print('download:', archive_path)
files.download(archive_path)